# Evaluation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import numpy as np
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [ ]:
env = r"D:\stackroute\2_AI-assisted-programming\learning_requirements\bosch\2026\5_advancedPE\code\config.env"
if load_dotenv(env):
    gemini_key = os.getenv("GEMINI_API_KEY")

# print(gemini_key)

In [8]:
# Main Gemini model used as an evaluator
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",temperature=0, api_key=gemini_key)

# Common evaluation result
class EvaluationResult(BaseModel):
    score: float = Field(ge=0.0, le=1.0, description="Evaluation score between 0 and 1")
    explanation: str

# Gemini with structured output
evaluator = llm.with_structured_output(EvaluationResult)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
relevancy_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an answer relevancy evaluator.

        Compare the user question and generated answer.

        Give a score between 0 and 1:
        - 1.0: The answer directly and completely addresses the question.
        - 0.5: The answer is partially relevant.
        - 0.0: The answer is unrelated.

        Ignore factual correctness. Evaluate only relevancy.
        """
    ),
    (
        "human",
        """
        Question:
        {question}

        Generated answer:
        {answer}
        """
    )
])

relevancy_chain = relevancy_prompt | evaluator

result = relevancy_chain.invoke({"question": "What are the benefits of regular exercise?",
    "answer": """
    Regular exercise improves cardiovascular health, strengthens muscles,
    helps manage weight and can reduce stress.
    """
})

print("Answer relevancy:", result.score)
print("Explanation:", result.explanation)

Answer relevancy: 1.0
Explanation: The answer directly lists several key benefits of regular exercise, fully addressing the user's question.


In [ ]:
# Task Completion
# Task completion checks whether all requested tasks were completed.

In [10]:
task_completion_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a task-completion evaluator.

        Determine whether the response completed every requirement.

        Give a score between 0 and 1:
        - 1.0: Every requirement was completed.
        - 0.5: Some requirements were completed.
        - 0.0: The task was not completed.

        Identify any missing requirements in the explanation.
        """
    ),
    (
        "human",
        """
        User task:
        {task}

        Generated response:
        {response}
        """
    )
])

task_completion_chain = task_completion_prompt | evaluator

result = task_completion_chain.invoke({
    "task": """
    List three benefits of cloud computing and give one example
    for each benefit.
    """,
    "response": """
    1. Scalability: A retailer can increase computing capacity during sales.
    2. Cost efficiency: A startup can avoid buying physical servers.
    3. Accessibility: Employees can access applications remotely.
    """
})

print("Task completion:", result.score)
print("Explanation:", result.explanation)

Task completion: 1.0
Explanation: The response successfully listed three benefits of cloud computing and provided a relevant example for each.


In [ ]:
# Prompt Alignment
# Prompt alignment checks whether the response follows the instructions, constraints and requested style.

In [11]:
alignment_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a prompt-alignment evaluator.

        Check whether the response follows all instructions and constraints
        in the original prompt.

        Give a score between 0 and 1:
        - 1.0: All instructions were followed.
        - 0.5: Some instructions were followed.
        - 0.0: The response ignored the instructions.

        Focus on instruction-following, not factual correctness.
        """
    ),
    (
        "human",
        """
        Original prompt:
        {original_prompt}

        Generated response:
        {response}
        """
    )
])

alignment_chain = alignment_prompt | evaluator

result = alignment_chain.invoke({
    "original_prompt": """
    Explain machine learning in exactly two sentences.
    Use simple language and do not use technical terms.
    """,
    "response": """
    Machine learning helps computers learn patterns from examples.
    It allows computers to make useful predictions without being given
    every rule.
    """
})

print("Prompt alignment:", result.score)
print("Explanation:", result.explanation)

Prompt alignment: 1.0
Explanation: The response follows all instructions: it is exactly two sentences, uses simple language, and avoids technical jargon.


In [ ]:
# Correctness

In [12]:
correctness_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a correctness evaluator.

        Compare the generated answer with the reference answer.

        Give a score between 0 and 1:
        - 1.0: Fully correct
        - 0.5: Partially correct
        - 0.0: Incorrect

        Minor differences in wording should not reduce the score.
        Focus on factual and logical correctness.
        """
    ),
    (
        "human",
        """
        Question:
        {question}

        Reference answer:
        {reference_answer}

        Generated answer:
        {generated_answer}
        """
    )
])

correctness_chain = correctness_prompt | evaluator

result = correctness_chain.invoke({
    "question": "What is the capital of Australia?",
    "reference_answer": "The capital of Australia is Canberra.",
    "generated_answer": "Canberra is the capital city of Australia."
})

print("Correctness:", result.score)
print("Explanation:", result.explanation)

Correctness: 1.0
Explanation: The generated answer correctly identifies Canberra as the capital of Australia, matching the factual content of the reference answer.


In [ ]:
# Semantic Similarity

In [14]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=gemini_key)

reference_answer = """
Exercise can improve heart health and reduce mental stress.
"""

generated_answer = """
Regular physical activity supports cardiovascular fitness
and can help people feel less stressed.
"""

# Convert both answers into embedding vectors
reference_vector = embeddings.embed_query(reference_answer)
generated_vector = embeddings.embed_query(generated_answer)

# Convert to NumPy arrays
reference_vector = np.array(reference_vector)
generated_vector = np.array(generated_vector)


# Calculate cosine similarity
similarity = np.dot(reference_vector,generated_vector
) / (
    np.linalg.norm(reference_vector)
    * np.linalg.norm(generated_vector)
)

print("Semantic similarity:", round(float(similarity), 4))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Semantic similarity: 0.809
